# LeetCode 559 — Maximum Depth of N-ary Tree bằng Python

**Link đề:** https://leetcode.com/problems/maximum-depth-of-n-ary-tree/
**Độ khó:** Easy · **Chủ đề:** Tree, Depth-First Search, Breadth-First Search

```python
# Definition for a Node.
# class Node:
#     def __init__(self, val: Optional[int] = None, children: Optional[List['Node']] = None):
#         self.val = val
#         self.children = children

class Solution:
    def maxDepth(self, root: 'Node') -> int:
        ...
```

## Mục tiêu học
1. Hiểu **cây N-phân** (N-ary tree): mỗi node có **list** con `children` thay vì chỉ `left`/`right`.
2. Đọc được cách LeetCode **tuần tự hóa** cây N-phân (level order, mỗi nhóm con kết thúc bằng `null`) và tự viết hàm dựng cây.
3. Tính **độ sâu lớn nhất** bằng 4 cách: DFS đệ quy → DFS lặp có độ sâu → BFS theo tầng → đệ quy một dòng với `max(..., default=0)`.
4. Trace call stack và queue theo tầng; chứng minh đúng; phân tích Time/Space; xử lý cây rỗng và cây sâu 1000 tầng.

## 1. Đề bài

Cho một cây N-phân, tìm **độ sâu lớn nhất** (maximum depth).

Độ sâu lớn nhất = **số node** trên đường dài nhất từ root xuống lá xa nhất.

```text
Ví dụ 1:            1
                 /  |  \
                3   2   4
               / \
              5   6
```

Đường dài nhất `1 → 3 → 5` (hoặc `1 → 3 → 6`) có 3 node → đáp án **3**.

| `root` (tuần tự hóa của LeetCode) | Output |
|---|---:|
| `[1,null,3,2,4,null,5,6]` | 3 |
| `[1,null,2,3,4,5,null,null,6,7,null,8,null,9,10,null,null,11,null,12,null,13,null,null,14]` | 5 |
| `[]` (cây rỗng) | 0 |

**Constraints:** tổng số node trong `[0, 10^4]`; độ sâu cây ≤ `1000`.

Lưu ý: **cây có thể rỗng** (`root = None`) → đáp án 0.

## 2. Nền tảng: class `Node` với list `children`

Khác cây nhị phân, mỗi node có thể có **bất kỳ số con** nào, lưu trong list `children`. Lá có `children` rỗng `[]` (trên LeetCode đôi khi là `None`, nên code an toàn sẽ viết `node.children or []`).

In [1]:
from typing import Optional, List
from collections import deque

class Node:
    def __init__(self, val: Optional[int] = None, children: Optional[List["Node"]] = None):
        self.val = val
        self.children = children if children is not None else []

# Tự tay dựng ví dụ 1
root = Node(1, [Node(3, [Node(5), Node(6)]), Node(2), Node(4)])
print("root:", root.val)
print("con của root:", [c.val for c in root.children])
print("con của 3:", [c.val for c in root.children[0].children])
print("node 2 là lá?", root.children[1].children == [])

root: 1
con của root: [3, 2, 4]
con của 3: [5, 6]
node 2 là lá? True


Vì sao `children if children is not None else []` mà không viết mặc định `children=[]`? Trong Python, giá trị mặc định là **list dùng chung** cho mọi lần gọi — nếu một node `append` vào đó, mọi node khác cũng "thấy" con mới. Đây là lỗi kinh điển "mutable default argument".

In [2]:
class BadNode:
    def __init__(self, val, children=[]):     # nguy hiểm
        self.val = val
        self.children = children

a = BadNode("a")
c = BadNode("c")
a.children.append(BadNode("x"))
print("con của c (đáng lẽ rỗng):", [n.val for n in c.children])

con của c (đáng lẽ rỗng): ['x']


### 2.1. Đọc cách tuần tự hóa của LeetCode

`[1,null,3,2,4,null,5,6]` đọc như sau (theo tầng, dùng queue):

1. Phần tử đầu là root `1`, sau đó là `null` (ngăn cách).
2. Lấy node đầu queue (`1`), đọc các giá trị đến khi gặp `null` → con của `1` là `3, 2, 4`.
3. Lấy node tiếp (`3`) → con là `5, 6` (hết list).
4. Các node còn lại không có con.

Mỗi **nhóm con** của một node kết thúc bằng một `null`. Node không có con → gặp `null` ngay.

In [3]:
def build_nary(values: list) -> Optional[Node]:
    if not values:
        return None
    root = Node(values[0])
    queue = deque([root])
    i = 2                                   # bỏ root và null ngăn cách đầu tiên
    while queue and i <= len(values):
        node = queue.popleft()
        while i < len(values) and values[i] is not None:
            child = Node(values[i])
            node.children.append(child)
            queue.append(child)
            i += 1
        i += 1                              # bỏ qua null kết thúc nhóm con
    return root

def draw(node: Optional[Node], depth: int = 0) -> None:
    if node is None:
        print("(cây rỗng)")
        return
    print("    " * depth + str(node.val))
    for child in node.children:
        draw(child, depth + 1)

null = None
ex1 = build_nary([1, null, 3, 2, 4, null, 5, 6])
ex2 = build_nary([1, null, 2, 3, 4, 5, null, null, 6, 7, null, 8, null, 9, 10, null, null, 11, null, 12, null, 13, null, null, 14])
draw(ex1)
print("----")
draw(ex2)

1
    3
        5
        6
    2
    4
----
1
    2
    3
        6
        7
            11
                14
    4
        8
            12
    5
        9
            13
        10


`deque` (double-ended queue) trong `collections`: `append` thêm cuối, `popleft` lấy đầu trong `O(1)` (list thường `pop(0)` tốn `O(n)`). Đặt `null = None` để chép nguyên văn ví dụ LeetCode.

Hàm `draw` in mỗi node trên một dòng, thụt lề 4 dấu cách cho mỗi tầng: nhìn số lần thụt lề lớn nhất + 1 là ra độ sâu.

## 3. Định nghĩa đệ quy của độ sâu

Ở cây nhị phân: `depth(node) = 1 + max(depth(left), depth(right))`.

Ở cây N-phân, thay "trái/phải" bằng "**mọi con**":

```text
depth(None)  = 0
depth(lá)    = 1
depth(node)  = 1 + max(depth(c) for c in node.children)
```

Ví dụ 1: `depth(5) = depth(6) = 1`; `depth(3) = 1 + max(1, 1) = 2`; `depth(2) = depth(4) = 1`; `depth(1) = 1 + max(2, 1, 1) = 3`.

In [4]:
print(max([2, 1, 1]))
try:
    print(max([]))
except ValueError as e:
    print("max([]) lỗi:", e)
print(max([], default=0))   # an toàn với list rỗng

2
max([]) lỗi: max() iterable argument is empty
0


**Bẫy quan trọng:** với lá, `node.children` rỗng → `max([])` báo `ValueError`. Hai cách xử lý:
1. Kiểm tra riêng lá, hoặc khởi tạo `best = 0` rồi duyệt con.
2. Dùng `max(..., default=0)`.

## 4. Cách 1 — DFS đệ quy (lời giải chính)

Viết từng phần:
1. Base case: `root is None` → 0.
2. Duyệt mọi con, lấy độ sâu lớn nhất `best` (bắt đầu từ 0 nên lá tự cho `best = 0`).
3. Trả `1 + best`.

In [5]:
def max_depth_dfs(root: Optional[Node]) -> int:
    if root is None:
        return 0
    best = 0
    for child in root.children:
        best = max(best, max_depth_dfs(child))
    return 1 + best

print(max_depth_dfs(ex1))   # 3
print(max_depth_dfs(ex2))   # 5
print(max_depth_dfs(None))  # 0

3
5
0


## 5. Trace call stack

Thứ tự **hoàn thành** (post-order) ở ví dụ 1: 5, 6, 3, 2, 4, 1.

| Node xong | độ sâu các con | `best` | trả về |
|---:|---|---:|---:|
| 5 | (không con) | 0 | 1 |
| 6 | (không con) | 0 | 1 |
| 3 | [1, 1] | 1 | 2 |
| 2 | (không con) | 0 | 1 |
| 4 | (không con) | 0 | 1 |
| 1 | [2, 1, 1] | 2 | **3** |

In [6]:
def trace_depth(node: Optional[Node], level: int = 0) -> int:
    pad = "  " * level
    if node is None:
        return 0
    print(f"{pad}vào {node.val} (con: {[c.val for c in node.children]})")
    child_depths = [trace_depth(c, level + 1) for c in node.children]
    best = max(child_depths, default=0)
    print(f"{pad}xong {node.val}: độ sâu con = {child_depths} -> trả {1 + best}")
    return 1 + best

print("Đáp án:", trace_depth(ex1))

vào 1 (con: [3, 2, 4])
  vào 3 (con: [5, 6])
    vào 5 (con: [])
    xong 5: độ sâu con = [] -> trả 1
    vào 6 (con: [])
    xong 6: độ sâu con = [] -> trả 1
  xong 3: độ sâu con = [1, 1] -> trả 2
  vào 2 (con: [])
  xong 2: độ sâu con = [] -> trả 1
  vào 4 (con: [])
  xong 4: độ sâu con = [] -> trả 1
xong 1: độ sâu con = [2, 1, 1] -> trả 3
Đáp án: 3


### Code nộp LeetCode — DFS đệ quy

Dùng `root.children or []` để an toàn khi LeetCode để `children = None` ở lá.

In [7]:
class Solution:
    def maxDepth(self, root: "Node") -> int:
        if root is None:
            return 0
        best = 0
        for child in root.children or []:
            best = max(best, self.maxDepth(child))
        return 1 + best

sol = Solution()
print(sol.maxDepth(ex1), sol.maxDepth(ex2), sol.maxDepth(None))

3 5 0


`a or b` trả về `a` nếu `a` "truthy", ngược lại trả `b`. `None or []` → `[]`; `[x] or []` → `[x]`.

### Độ sâu đệ quy

Đề nói độ sâu ≤ 1000. Python mặc định cho khoảng 1000 tầng gọi hàm; cây sâu đúng 1000 có thể chạm giới hạn (vì còn vài khung gọi khác). An toàn: `sys.setrecursionlimit(5000)` hoặc dùng cách lặp (mục 6, 7).

In [8]:
import sys

def build_chain_nary(n: int) -> Node:
    root = Node(1)
    cur = root
    for v in range(2, n + 1):
        nxt = Node(v)
        cur.children.append(nxt)
        cur = nxt
    return root

deep = build_chain_nary(1000)
print("Giới hạn đệ quy:", sys.getrecursionlimit())
try:
    print("DFS đệ quy:", max_depth_dfs(deep))
except RecursionError:
    print("RecursionError với cây sâu 1000")
sys.setrecursionlimit(5000)
print("Sau khi tăng giới hạn:", max_depth_dfs(deep))

Giới hạn đệ quy: 1000
RecursionError với cây sâu 1000
Sau khi tăng giới hạn: 1000


## 6. Cách 2 — BFS theo tầng (level order)

Ý tưởng: duyệt cây **từng tầng một** bằng queue. Mỗi lần xử lý xong một tầng, tăng `depth` thêm 1. Số tầng = độ sâu lớn nhất.

Kỹ thuật "xử lý trọn một tầng": lấy `size = len(queue)` **trước** vòng trong, rồi `popleft` đúng `size` lần — các con mới thêm vào thuộc tầng sau.

In [9]:
def max_depth_bfs(root: Optional[Node]) -> int:
    if root is None:
        return 0
    depth = 0
    queue = deque([root])
    while queue:
        depth += 1
        for _ in range(len(queue)):      # đúng số node của tầng hiện tại
            node = queue.popleft()
            for child in node.children:
                queue.append(child)
    return depth

print(max_depth_bfs(ex1), max_depth_bfs(ex2), max_depth_bfs(None), max_depth_bfs(deep))

3 5 0 1000


In [10]:
def trace_bfs(root: Optional[Node]) -> int:
    if root is None:
        return 0
    depth = 0
    queue = deque([root])
    while queue:
        depth += 1
        level = [n.val for n in queue]
        for _ in range(len(queue)):
            node = queue.popleft()
            queue.extend(node.children)
        print(f"tầng {depth}: {level} -> queue sau: {[n.val for n in queue]}")
    print("Độ sâu =", depth)
    return depth

trace_bfs(ex2)

tầng 1: [1] -> queue sau: [2, 3, 4, 5]
tầng 2: [2, 3, 4, 5] -> queue sau: [6, 7, 8, 9, 10]
tầng 3: [6, 7, 8, 9, 10] -> queue sau: [11, 12, 13]
tầng 4: [11, 12, 13] -> queue sau: [14]
tầng 5: [14] -> queue sau: []
Độ sâu = 5


5

`queue.extend(list)` thêm **tất cả** phần tử của list vào cuối — gọn hơn vòng `for` + `append`. `range(len(queue))` được tính **một lần** khi bắt đầu vòng `for`, nên các con thêm vào giữa chừng không làm vòng chạy thêm.

### Code nộp LeetCode — BFS

In [11]:
class Solution:
    def maxDepth(self, root: "Node") -> int:
        if root is None:
            return 0
        depth = 0
        queue = deque([root])
        while queue:
            depth += 1
            for _ in range(len(queue)):
                node = queue.popleft()
                queue.extend(node.children or [])
        return depth

print(Solution().maxDepth(ex2))

5


## 7. Cách 3 — DFS lặp bằng stack, mang theo độ sâu

Thay vì để call stack nhớ độ sâu, ta tự đẩy cặp `(node, độ_sâu_của_node)` vào stack. Mỗi lần lấy ra, cập nhật `best`, rồi đẩy các con với độ sâu `+1`.

Thứ tự duyệt không quan trọng (ta chỉ cần max), nên stack (LIFO) hay queue đều được.

In [12]:
def max_depth_stack(root: Optional[Node]) -> int:
    if root is None:
        return 0
    best = 0
    stack = [(root, 1)]
    while stack:
        node, d = stack.pop()
        best = max(best, d)
        for child in node.children:
            stack.append((child, d + 1))
    return best

print(max_depth_stack(ex1), max_depth_stack(ex2), max_depth_stack(None), max_depth_stack(deep))

3 5 0 1000


In [13]:
def trace_stack(root):
    best, stack = 0, [(root, 1)]
    while stack:
        node, d = stack.pop()
        best = max(best, d)
        stack.extend((c, d + 1) for c in node.children)
        print(f"lấy {node.val} (d={d}) best={best} | stack={[(n.val, x) for n, x in stack]}")
    return best

trace_stack(ex1)

lấy 1 (d=1) best=1 | stack=[(3, 2), (2, 2), (4, 2)]
lấy 4 (d=2) best=2 | stack=[(3, 2), (2, 2)]
lấy 2 (d=2) best=2 | stack=[(3, 2)]
lấy 3 (d=2) best=2 | stack=[(5, 3), (6, 3)]
lấy 6 (d=3) best=3 | stack=[(5, 3)]
lấy 5 (d=3) best=3 | stack=[]


3

## 8. Cách 4 — Đệ quy một dòng

Ghép định nghĩa ở mục 3 với `max(..., default=0)` và **generator expression**:

In [14]:
def max_depth_oneliner(root: Optional[Node]) -> int:
    return 0 if root is None else 1 + max((max_depth_oneliner(c) for c in root.children), default=0)

print(max_depth_oneliner(ex1), max_depth_oneliner(ex2), max_depth_oneliner(None))

3 5 0


`A if điều_kiện else B` là **biểu thức điều kiện** (ternary). Ngoặc tròn bao generator là bắt buộc khi có thêm tham số `default=...`. Gọn, nhưng khó debug hơn Cách 1 — chỉ nên dùng khi đã hiểu rõ.

## 9. Vì sao đúng?

**Cách 1 (quy nạp theo cây con):**
- Cây rỗng: 0 node → độ sâu 0 ✔.
- Giả sử `max_depth_dfs(c)` đúng cho mọi con `c`. Mọi đường từ `node` xuống lá đều đi qua đúng một con `c`, nên dài `1 + (độ dài đường trong cây con c)`. Lấy max trên mọi con → `1 + max(depth(c))` ✔. Lá: không có con, đường chỉ gồm chính nó → `1 + 0 = 1` ✔.

**Cách 2 (BFS) — invariant:** ở đầu vòng `while` thứ `t`, queue chứa **đúng** các node ở tầng `t`. Ban đầu queue = [root] (tầng 1). Xử lý hết tầng `t` và thêm mọi con → queue = tầng `t + 1`. Vòng dừng khi tầng rỗng → `depth` = số tầng có node = độ sâu lớn nhất ✔.

**Cách 3:** mỗi node được đẩy vào stack đúng một lần kèm độ sâu đúng của nó (độ sâu cha + 1) → `best` = max độ sâu mọi node = độ sâu của lá sâu nhất ✔.

## 10. So sánh và độ phức tạp

Gọi `n` = số node, `h` = độ sâu, `w` = số node nhiều nhất trong một tầng.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. DFS đệ quy | `O(n)` | `O(h)` call stack | **ngắn, tự nhiên** |
| 2. BFS theo tầng | `O(n)` | `O(w)` queue | không lo đệ quy; đếm tầng trực quan |
| 3. DFS lặp + độ sâu | `O(n)` | `O(n)` xấu nhất (stack chứa nhiều con) | không lo đệ quy |
| 4. Một dòng | `O(n)` | `O(h)` | gọn |

Mỗi node được thăm đúng một lần, và mỗi con được duyệt đúng một lần từ cha của nó ⇒ tổng `O(n)`.

## 11. Bộ test: mẫu đề, biên, ngẫu nhiên

In [15]:
tests = [
    ([1, null, 3, 2, 4, null, 5, 6], 3),
    ([1, null, 2, 3, 4, 5, null, null, 6, 7, null, 8, null, 9, 10, null, null, 11, null, 12, null, 13, null, null, 14], 5),
    ([], 0),
    ([1], 1),
    ([1, null, 2, 3, 4, 5, 6], 2),                    # một root, nhiều lá
    ([1, null, 2, null, 3, null, 4], 4),              # dây chuyền
]
methods = [max_depth_dfs, max_depth_bfs, max_depth_stack, max_depth_oneliner]
for values, expected in tests:
    t = build_nary(values)
    for f in methods:
        assert f(t) == expected, (f.__name__, values)
    print(f"{str(values)[:50]:<52} -> {expected}")
print("Test mẫu PASS!")

[1, None, 3, 2, 4, None, 5, 6]                       -> 3
[1, None, 2, 3, 4, 5, None, None, 6, 7, None, 8, N   -> 5
[]                                                   -> 0
[1]                                                  -> 1
[1, None, 2, 3, 4, 5, 6]                             -> 2
[1, None, 2, None, 3, None, 4]                       -> 4
Test mẫu PASS!


In [16]:
import random

def random_nary(n: int) -> Optional[Node]:
    if n == 0:
        return None
    nodes = [Node(0)]
    for v in range(1, n):
        parent = random.choice(nodes)
        child = Node(v)
        parent.children.append(child)
        nodes.append(child)
    return nodes[0]

random.seed(559)
for _ in range(300):
    t = random_nary(random.randint(0, 60))
    answers = {f(t) for f in methods}
    assert len(answers) == 1, answers
print("300 cây ngẫu nhiên: 4 cách cho cùng kết quả.")

300 cây ngẫu nhiên: 4 cách cho cùng kết quả.


## 12. Lỗi người mới hay gặp

### Lỗi 1: `max([])` ở lá

In [17]:
def wrong_max_empty(root):
    if root is None:
        return 0
    return 1 + max([wrong_max_empty(c) for c in root.children])   # lá -> max([])

try:
    wrong_max_empty(ex1)
except ValueError as e:
    print("ValueError:", e)

ValueError: max() iterable argument is empty


Sửa: `max(..., default=0)` hoặc khởi tạo `best = 0` rồi duyệt.

### Lỗi 2: Quên trường hợp cây rỗng

Gọi `root.children` khi `root is None` → `AttributeError`. Đề cho phép 0 node, nên **luôn** kiểm tra `root is None` trước.

In [18]:
def wrong_no_none_check(root):
    best = 0
    for c in root.children:
        best = max(best, wrong_no_none_check(c))
    return 1 + best

try:
    wrong_no_none_check(None)
except AttributeError as e:
    print("AttributeError:", e)

AttributeError: 'NoneType' object has no attribute 'children'


### Lỗi 3: BFS tăng `depth` cho mỗi node thay vì mỗi tầng

Nếu `depth += 1` nằm **trong** vòng lấy node, ta đếm **số node** chứ không phải số tầng.

In [19]:
def wrong_bfs(root):
    depth, queue = 0, deque([root])
    while queue:
        node = queue.popleft()
        depth += 1                      # SAI: tăng theo node
        queue.extend(node.children)
    return depth

print("sai:", wrong_bfs(ex1), "| đúng:", max_depth_bfs(ex1))

sai: 6 | đúng: 3


### Lỗi 4: Chỉ xét con đầu tiên / con cuối cùng

Nhầm với cây nhị phân, chỉ đệ quy vào `children[0]` → bỏ sót nhánh sâu hơn ở các con khác.

### Lỗi 5: `children=[]` làm tham số mặc định

Xem mục 2: mọi node dùng chung một list → cây bị "dính" con lạ.

## 13. Mẹo nhận dạng pattern

- "Độ sâu/chiều cao" của cây → `1 + max(độ sâu con)`; nhớ base case cây rỗng.
- Cây N-phân = cây nhị phân nhưng thay `left/right` bằng **vòng `for child in children`**.
- Cần thông tin **theo tầng** (số tầng, giá trị mỗi tầng, tầng nhỏ nhất...) → **BFS với `len(queue)`**.
- Sợ đệ quy quá sâu → stack mang theo `(node, depth)`.

## 14. Biến thể

1. **LeetCode 104 – Maximum Depth of Binary Tree**.
2. **LeetCode 111 – Minimum Depth of Binary Tree**: BFS dừng ở lá đầu tiên.
3. **LeetCode 429 – N-ary Tree Level Order Traversal**: BFS trả về giá trị từng tầng.
4. **LeetCode 589/590 – N-ary Tree Preorder/Postorder Traversal**.
5. **LeetCode 1522 – Diameter of N-Ary Tree**: lấy hai con sâu nhất.

In [20]:
# Biến thể 3 demo: level order của cây N-phân
def level_order(root):
    if root is None:
        return []
    res, queue = [], deque([root])
    while queue:
        level = []
        for _ in range(len(queue)):
            node = queue.popleft()
            level.append(node.val)
            queue.extend(node.children)
        res.append(level)
    return res

print(level_order(ex1))
print(level_order(ex2))

[[1], [3, 2, 4], [5, 6]]
[[1], [2, 3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13], [14]]


Số phần tử của `level_order(root)` chính là độ sâu lớn nhất — một cách kiểm tra chéo nữa.

## 15. Bài tập tự luyện

### Bài 1 (dễ)
Viết `count_nodes_nary(root)` đếm tổng số node.

In [21]:
def count_nodes_nary(root) -> int:
    # TODO
    pass

# assert count_nodes_nary(ex1) == 6
# assert count_nodes_nary(ex2) == 14
# assert count_nodes_nary(None) == 0

### Bài 2 (dễ)
Viết `count_leaves(root)` đếm số lá (node không có con).

In [22]:
def count_leaves(root) -> int:
    # TODO
    pass

# assert count_leaves(ex1) == 4     # 5, 6, 2, 4
# assert count_leaves(build_nary([1])) == 1

### Bài 3 (vừa)
Viết `min_depth_nary(root)`: số node trên đường ngắn nhất từ root tới **một lá**.

In [23]:
def min_depth_nary(root) -> int:
    # TODO: BFS, gặp lá đầu tiên thì trả về
    pass

# assert min_depth_nary(ex1) == 2
# assert min_depth_nary(ex2) == 2
# assert min_depth_nary(None) == 0

### Bài 4 (vừa)
Viết `deepest_values(root)`: list giá trị của các node ở **tầng sâu nhất** (theo thứ tự trái → phải).

In [24]:
def deepest_values(root) -> list:
    # TODO
    pass

# assert deepest_values(ex1) == [5, 6]
# assert deepest_values(ex2) == [14]

### Bài 5 (khó hơn)
Viết `longest_root_to_leaf_path(root)`: trả về **list giá trị** trên một đường dài nhất từ root xuống lá (nếu nhiều đường bằng nhau, lấy đường gặp trước theo thứ tự con).

In [25]:
def longest_root_to_leaf_path(root) -> list:
    # TODO
    pass

# assert longest_root_to_leaf_path(ex1) == [1, 3, 5]
# assert longest_root_to_leaf_path(ex2) == [1, 3, 7, 11, 14]

## 16. Gợi ý 3 mức

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Đệ quy giống độ sâu | Thay `max` bằng `sum` | `1 + sum(count(c) for c in children)` |
| 2 | Lá: `children` rỗng | Lá trả 1 | Không phải lá: tổng số lá các con |
| 3 | BFS theo tầng | Gặp node không con → trả `depth` | Đừng dùng `min(...)` mà quên lá |
| 4 | Dùng `level_order` | Lấy tầng cuối | `level_order(root)[-1]` |
| 5 | Đệ quy trả về **list đường** | So độ dài các đường con | Chỉ thay khi **dài hơn hẳn** để giữ đường gặp trước |

## 17. Đáp án tham khảo (tự làm trước rồi mới xem)

In [26]:
def answer_count_nodes_nary(root) -> int:
    if root is None:
        return 0
    return 1 + sum(answer_count_nodes_nary(c) for c in root.children)

def answer_count_leaves(root) -> int:
    if root is None:
        return 0
    if not root.children:
        return 1
    return sum(answer_count_leaves(c) for c in root.children)

def answer_min_depth_nary(root) -> int:
    if root is None:
        return 0
    depth, queue = 0, deque([root])
    while queue:
        depth += 1
        for _ in range(len(queue)):
            node = queue.popleft()
            if not node.children:
                return depth
            queue.extend(node.children)
    return depth

def answer_deepest_values(root) -> list:
    levels = level_order(root)
    return levels[-1] if levels else []

def answer_longest_root_to_leaf_path(root) -> list:
    if root is None:
        return []
    best = []
    for c in root.children:
        p = answer_longest_root_to_leaf_path(c)
        if len(p) > len(best):
            best = p
    return [root.val] + best

assert answer_count_nodes_nary(ex1) == 6
assert answer_count_nodes_nary(ex2) == 14
assert answer_count_nodes_nary(None) == 0
assert answer_count_leaves(ex1) == 4
assert answer_count_leaves(build_nary([1])) == 1
assert answer_min_depth_nary(ex1) == 2
assert answer_min_depth_nary(ex2) == 2
assert answer_min_depth_nary(None) == 0
assert answer_deepest_values(ex1) == [5, 6]
assert answer_deepest_values(ex2) == [14]
assert answer_longest_root_to_leaf_path(ex1) == [1, 3, 5]
assert answer_longest_root_to_leaf_path(ex2) == [1, 3, 7, 11, 14]
for _ in range(100):
    t = random_nary(random.randint(1, 40))
    assert len(answer_longest_root_to_leaf_path(t)) == max_depth_dfs(t)
print("Đáp án chạy PASS toàn bộ test!")

Đáp án chạy PASS toàn bộ test!


## 18. Template nhớ nhanh và checklist

```python
class Solution:
    def maxDepth(self, root: "Node") -> int:
        if root is None:
            return 0
        best = 0
        for child in root.children or []:
            best = max(best, self.maxDepth(child))
        return 1 + best
```

BFS đếm tầng:

```python
depth, q = 0, deque([root]) if root else deque()
while q:
    depth += 1
    for _ in range(len(q)):
        q.extend(q.popleft().children or [])
return depth
```

### Checklist trước khi submit
- [ ] Xử lý `root is None` → 0.
- [ ] Lá không gây `max([])` (dùng `best = 0` hoặc `default=0`).
- [ ] Duyệt **mọi** con, không chỉ con đầu.
- [ ] BFS: tăng `depth` theo **tầng**, dùng `len(queue)` chốt trước.
- [ ] Nhớ cây có thể sâu 1000 → cân nhắc bản lặp.